# Results: Consolidação dos resultados (Baselines + Cross-dataset)

Calcula as métricas (Dice global, LWD) sobre WT/TC/ET
a partir das predições no disco. Considera os 8 cenários:

**In-domain** (4):
- Baseline_2020 testado em BraTS2020
- Baseline_2024 testado em BraTS2024
- MoME_2020 testado em BraTS2020
- MoME_2024 testado em BraTS2024

**Cross-domain** (4):
- nnUNet 2020 → BraTS2024
- nnUNet 2024 → BraTS2020
- MoME 2020 → BraTS2024
- MoME 2024 → BraTS2020

In [1]:
import os, json
from pathlib import Path
import numpy as np
import nibabel as nib
import pandas as pd
from tqdm import tqdm
from scipy.ndimage import label as cc_label

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

SPLITS_2020 = Path(r"D:\master_experiments\data\splits\BraTS2020_Splits")
SPLITS_2024 = Path(r"D:\master_experiments\data\splits\BraTS2024_Splits")

NNUNET_PRED_2020_IN = Path(r"D:\nnUNet_preds\brats2020_test_pred")
NNUNET_PRED_2024_IN = Path(r"D:\nnUNet_preds\brats2024_test_pred")
MOME_PRED_2020_IN   = Path(r"D:\master_experiments\models_configs\MoME_BraTS2020\predictions_test")
MOME_PRED_2024_IN   = Path(r"D:\master_experiments\models_configs\MoME_BraTS2024\predictions_test")

CROSS_BASE = Path(r"D:\master_experiments\models_configs\cross_eval")
NNUNET_CROSS_A = CROSS_BASE / "nnUNet" / "preds_2020_on_2024"  # modelo 2020 -> teste 2024
NNUNET_CROSS_B = CROSS_BASE / "nnUNet" / "preds_2024_on_2020"  # modelo 2024 -> teste 2020
MOME_CROSS_A   = CROSS_BASE / "MoME"   / "preds_2020_on_2024"
MOME_CROSS_B   = CROSS_BASE / "MoME"   / "preds_2024_on_2020"

RESULTS_DIR = Path(r"D:\master_experiments\models_configs\results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Paths OK")

Paths OK


In [2]:
def case_dir_2020(cid): return SPLITS_2020 / "test" / cid
def case_dir_2024(cid): return SPLITS_2024 / "test" / cid

def find_seg(d):
    for cand in [d / "seg.nii.gz", d / "seg.nii"]:
        if cand.exists(): return cand
    cands = sorted(d.glob("*seg*.nii*"))
    if not cands:
        raise FileNotFoundError(f"seg nao encontrada em {d}")
    return cands[0]

def load_arr(p):
    return np.asanyarray(nib.load(str(p)).dataobj)

def load_seg_2020(cid):
    data = load_arr(find_seg(case_dir_2020(cid))).astype(np.int16)
    data[data == 4] = 3
    return data

def load_seg_2024(cid):
    return load_arr(find_seg(case_dir_2024(cid))).astype(np.int16)

def dice(a, b):
    inter = np.count_nonzero(a & b)
    denom = np.count_nonzero(a) + np.count_nonzero(b)
    return 1.0 if denom == 0 else (2.0 * inter / denom)

LW_MIN_SIZE = 50

def lesion_wise_dice(gt_mask, pr_mask, min_size=LW_MIN_SIZE):
    structure = np.ones((3, 3, 3), dtype=int)
    if not gt_mask.any() and not pr_mask.any():
        return 1.0
    gt_lab, n_gt = cc_label(gt_mask.astype(bool), structure=structure)
    pr_lab, n_pr = cc_label(pr_mask.astype(bool), structure=structure)
    dice_list = []
    matched_pred = set()
    for g in range(1, n_gt + 1):
        gt_l = (gt_lab == g)
        if gt_l.sum() < min_size: continue
        overlap_ids = np.unique(pr_lab[gt_l])
        overlap_ids = overlap_ids[overlap_ids > 0]
        if len(overlap_ids) == 0:
            dice_list.append(0.0)
        else:
            pr_match = np.isin(pr_lab, overlap_ids)
            inter = np.logical_and(gt_l, pr_match).sum()
            denom = gt_l.sum() + pr_match.sum()
            dice_list.append(2.0 * inter / denom if denom > 0 else 0.0)
            matched_pred.update(int(p) for p in overlap_ids)
    for p in range(1, n_pr + 1):
        if p in matched_pred: continue
        pr_l = (pr_lab == p)
        if pr_l.sum() < min_size: continue
        dice_list.append(0.0)
    return float(np.mean(dice_list)) if dice_list else 1.0

def evaluate_run(name, test_ids, gt_loader, pred_dir, suppress_4=False):
    rows, lwd_rows = [], []
    for cid in tqdm(test_ids, desc=f"Avaliando {name}"):
        gt = gt_loader(cid)
        pr = load_arr(pred_dir / f"{cid}.nii.gz").astype(np.int16)
        if suppress_4:
            pr[pr == 4] = 0

        gt_wt = (gt == 1) | (gt == 2) | (gt == 3)
        pr_wt = (pr == 1) | (pr == 2) | (pr == 3)
        gt_tc = (gt == 1) | (gt == 3)
        pr_tc = (pr == 1) | (pr == 3)
        gt_et = (gt == 3)
        pr_et = (pr == 3)

        rows.append({
            "id": cid,
            "dice_WT": dice(gt_wt, pr_wt),
            "dice_TC": dice(gt_tc, pr_tc),
            "dice_ET": dice(gt_et, pr_et),
        })
        lwd_rows.append({
            "id": cid,
            "lwd_WT": lesion_wise_dice(gt_wt, pr_wt),
            "lwd_TC": lesion_wise_dice(gt_tc, pr_tc),
            "lwd_ET": lesion_wise_dice(gt_et, pr_et),
        })

    return pd.DataFrame(rows), pd.DataFrame(lwd_rows)

with open(SPLITS_2020 / "splits_metadata.json") as f:
    test_ids_2020 = json.load(f)["ids"]["test"]
with open(SPLITS_2024 / "splits_metadata.json") as f:
    test_ids_2024 = json.load(f)["ids"]["test"]
print(f"Test 2020: {len(test_ids_2020)} | Test 2024: {len(test_ids_2024)}")

Test 2020: 53 | Test 2024: 203


In [3]:
# Definicao dos 8 cenarios (4 in-domain + 4 cross)

SCENARIOS = [
    # (name, test_ids, gt_loader, pred_dir, suppress_4)
    ("Baseline_2020 in-domain",      test_ids_2020, load_seg_2020, NNUNET_PRED_2020_IN, False),
    ("Baseline_2024 in-domain",      test_ids_2024, load_seg_2024, NNUNET_PRED_2024_IN, False),
    ("MoME_2020 in-domain",          test_ids_2020, load_seg_2020, MOME_PRED_2020_IN,   False),
    ("MoME_2024 in-domain",          test_ids_2024, load_seg_2024, MOME_PRED_2024_IN,   False),
    ("nnUNet 2020 -> 2024 (cross)",  test_ids_2024, load_seg_2024, NNUNET_CROSS_A,      False),
    ("nnUNet 2024 -> 2020 (cross)",  test_ids_2020, load_seg_2020, NNUNET_CROSS_B,      True),
    ("MoME 2020 -> 2024 (cross)",    test_ids_2024, load_seg_2024, MOME_CROSS_A,        False),
    ("MoME 2024 -> 2020 (cross)",    test_ids_2020, load_seg_2020, MOME_CROSS_B,        False),
]

print(f"Total cenarios: {len(SCENARIOS)}")
for name, tids, _, pdir, sup in SCENARIOS:
    print(f"  - {name:35s} | n={len(tids):3d} | preds={pdir.name} | suppress_4={sup}")

Total cenarios: 8
  - Baseline_2020 in-domain             | n= 53 | preds=brats2020_test_pred | suppress_4=False
  - Baseline_2024 in-domain             | n=203 | preds=brats2024_test_pred | suppress_4=False
  - MoME_2020 in-domain                 | n= 53 | preds=predictions_test | suppress_4=False
  - MoME_2024 in-domain                 | n=203 | preds=predictions_test | suppress_4=False
  - nnUNet 2020 -> 2024 (cross)         | n=203 | preds=preds_2020_on_2024 | suppress_4=False
  - nnUNet 2024 -> 2020 (cross)         | n= 53 | preds=preds_2024_on_2020 | suppress_4=True
  - MoME 2020 -> 2024 (cross)           | n=203 | preds=preds_2020_on_2024 | suppress_4=False
  - MoME 2024 -> 2020 (cross)           | n= 53 | preds=preds_2024_on_2020 | suppress_4=False


In [4]:
# Roda avaliacao de cada cenario
results = {} 

for name, tids, gt_loader, pred_dir, suppress in SCENARIOS:
    df_dice, df_lwd = evaluate_run(name, tids, gt_loader, pred_dir, suppress_4=suppress)
    safe = name.replace(" ", "_").replace("(","").replace(")","").replace("->","to")
    df_dice.to_csv(RESULTS_DIR / f"{safe}_dice.csv", index=False)
    df_lwd.to_csv(RESULTS_DIR / f"{safe}_lwd.csv", index=False)
    results[name] = {"dice": df_dice, "lwd": df_lwd}

print("\nAvaliacoes concluidas. CSVs salvos em:", RESULTS_DIR)

Avaliando MoME 2024 -> 2020 (cross): 100%|██████████| 53/53 [01:02<00:00,  1.18s/it]


Avaliacoes concluidas. CSVs salvos em: D:\master_experiments\models_configs\results


In [5]:
# Constroi tabela master com Dice global e LWD por cenario (medias)

summary_rows = []
for name in [s[0] for s in SCENARIOS]:
    d = results[name]["dice"]
    l = results[name]["lwd"]
    summary_rows.append({
        "cenario":  name,
        "n":        len(d),
        "Dice_WT":  d["dice_WT"].mean(),
        "Dice_TC":  d["dice_TC"].mean(),
        "Dice_ET":  d["dice_ET"].mean(),
        "LWD_WT":   l["lwd_WT"].mean(),
        "LWD_TC":   l["lwd_TC"].mean(),
        "LWD_ET":   l["lwd_ET"].mean(),
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(RESULTS_DIR / "summary_all.csv", index=False)
df_summary.round(4)

,cenario,n,Dice_WT,Dice_TC,Dice_ET,LWD_WT,LWD_TC,LWD_ET
0,Baseline_2020 in-domain,53,0.9112,0.9013,0.8369,0.7151,0.8623,0.7702
1,Baseline_2024 in-domain,203,0.8968,0.7017,0.7196,0.6855,0.7128,0.7240
2,MoME_2020 in-domain,53,0.8901,0.8637,0.7869,0.6153,0.7489,0.7001
3,MoME_2024 in-domain,203,0.8455,0.5133,0.5235,0.5992,0.5199,0.5268
4,nnUNet 2020 -> 2024 (cross),203,0.7414,0.4426,0.6045,0.5258,0.4111,0.6202
5,nnUNet 2024 -> 2020 (cross),53,0.8811,0.7825,0.8127,0.6623,0.7462,0.7512
6,MoME 2020 -> 2024 (cross),203,0.7411,0.3787,0.4922,0.5074,0.3144,0.4996
7,MoME 2024 -> 2020 (cross),53,0.8320,0.7168,0.7706,0.5543,0.5518,0.5714


In [6]:
# Tabela 1: In-domain (4 cenarios) — comparacao Baseline vs MoME por dataset

in_domain = df_summary[df_summary["cenario"].str.contains("in-domain")].copy().reset_index(drop=True)
in_domain.round(4)

,cenario,n,Dice_WT,Dice_TC,Dice_ET,LWD_WT,LWD_TC,LWD_ET
0,Baseline_2020 in-domain,53,0.9112,0.9013,0.8369,0.7151,0.8623,0.7702
1,Baseline_2024 in-domain,203,0.8968,0.7017,0.7196,0.6855,0.7128,0.7240
2,MoME_2020 in-domain,53,0.8901,0.8637,0.7869,0.6153,0.7489,0.7001
3,MoME_2024 in-domain,203,0.8455,0.5133,0.5235,0.5992,0.5199,0.5268


In [7]:
# Tabela 2: Cross-domain (4 cenarios) — generalizacao entre datasets

cross = df_summary[df_summary["cenario"].str.contains("cross")].copy().reset_index(drop=True)
cross.round(4)

,cenario,n,Dice_WT,Dice_TC,Dice_ET,LWD_WT,LWD_TC,LWD_ET
0,nnUNet 2020 -> 2024 (cross),203,0.7414,0.4426,0.6045,0.5258,0.4111,0.6202
1,nnUNet 2024 -> 2020 (cross),53,0.8811,0.7825,0.8127,0.6623,0.7462,0.7512
2,MoME 2020 -> 2024 (cross),203,0.7411,0.3787,0.4922,0.5074,0.3144,0.4996
3,MoME 2024 -> 2020 (cross),53,0.8320,0.7168,0.7706,0.5543,0.5518,0.5714


In [8]:
# Tabela 3: Delta cross vs in-domain por arquitetura
# Quanto cada modelo perde quando aplicado em dominio diferente

def get(name, col):
    return df_summary.loc[df_summary["cenario"] == name, col].values[0]

deltas = []
for arch, in_2020, in_2024, cr_to24, cr_to20 in [
    ("nnUNet", "Baseline_2020 in-domain", "Baseline_2024 in-domain",
     "nnUNet 2020 -> 2024 (cross)", "nnUNet 2024 -> 2020 (cross)"),
    ("MoME",   "MoME_2020 in-domain",     "MoME_2024 in-domain",
     "MoME 2020 -> 2024 (cross)",   "MoME 2024 -> 2020 (cross)"),
]:
    for direction, src_name, cross_name in [
        ("2020 -> 2024", in_2020, cr_to24), 
        ("2024 -> 2020", in_2024, cr_to20), 
    ]:
        for metric in ["Dice_WT", "Dice_TC", "Dice_ET", "LWD_WT", "LWD_TC", "LWD_ET"]:
            deltas.append({
                "arquitetura": arch,
                "direcao":     direction,
                "metrica":     metric,
                "in_domain":   get(src_name,   metric),
                "cross":       get(cross_name, metric),
                "delta":       get(cross_name, metric) - get(src_name, metric),
                "delta_pct":   100.0 * (get(cross_name, metric) - get(src_name, metric)) / max(get(src_name, metric), 1e-8),
            })

df_deltas = pd.DataFrame(deltas)
df_deltas.to_csv(RESULTS_DIR / "summary_deltas.csv", index=False)
df_deltas.round(4)

,arquitetura,direcao,metrica,in_domain,cross,delta,delta_pct
0,nnUNet,2020 -> 2024,Dice_WT,0.9112,0.7414,-0.1697,-18.6251
1,nnUNet,2020 -> 2024,Dice_TC,0.9013,0.4426,-0.4588,-50.8992
2,nnUNet,2020 -> 2024,Dice_ET,0.8369,0.6045,-0.2324,-27.7687
3,nnUNet,2020 -> 2024,LWD_WT,0.7151,0.5258,-0.1893,-26.4686
4,nnUNet,2020 -> 2024,LWD_TC,0.8623,0.4111,-0.4512,-52.3195
5,nnUNet,2020 -> 2024,LWD_ET,0.7702,0.6202,-0.1501,-19.4827
6,nnUNet,2024 -> 2020,Dice_WT,0.8968,0.8811,-0.0157,-1.7474
7,nnUNet,2024 -> 2020,Dice_TC,0.7017,0.7825,0.0807,11.5060
8,nnUNet,2024 -> 2020,Dice_ET,0.7196,0.8127,0.0932,12.9456
9,nnUNet,2024 -> 2020,LWD_WT,0.6855,0.6623,-0.0231,-3.3768


In [9]:
# Tabela 4: Comparacao direta nnUNet vs MoME em cada cenario
# Mostra qual abordagem performa melhor em cada combinacao

comparisons = []
for label, nn_name, mome_name in [
    ("BraTS2020 in-domain",      "Baseline_2020 in-domain",     "MoME_2020 in-domain"),
    ("BraTS2024 in-domain",      "Baseline_2024 in-domain",     "MoME_2024 in-domain"),
    ("Cross 2020 -> 2024",        "nnUNet 2020 -> 2024 (cross)", "MoME 2020 -> 2024 (cross)"),
    ("Cross 2024 -> 2020",        "nnUNet 2024 -> 2020 (cross)", "MoME 2024 -> 2020 (cross)"),
]:
    for metric in ["Dice_WT", "Dice_TC", "Dice_ET", "LWD_WT", "LWD_TC", "LWD_ET"]:
        nn_val   = get(nn_name,   metric)
        mome_val = get(mome_name, metric)
        comparisons.append({
            "cenario":      label,
            "metrica":      metric,
            "nnUNet":       nn_val,
            "MoME":         mome_val,
            "diff":         mome_val - nn_val,
            "vencedor":     "nnUNet" if nn_val > mome_val else "MoME",
        })

df_compare = pd.DataFrame(comparisons)
df_compare.to_csv(RESULTS_DIR / "summary_compare.csv", index=False)
df_compare.round(4)

,cenario,metrica,nnUNet,MoME,diff,vencedor
0,BraTS2020 in-domain,Dice_WT,0.9112,0.8901,-0.0210,nnUNet
1,BraTS2020 in-domain,Dice_TC,0.9013,0.8637,-0.0377,nnUNet
2,BraTS2020 in-domain,Dice_ET,0.8369,0.7869,-0.0500,nnUNet
3,BraTS2020 in-domain,LWD_WT,0.7151,0.6153,-0.0998,nnUNet
4,BraTS2020 in-domain,LWD_TC,0.8623,0.7489,-0.1134,nnUNet
5,BraTS2020 in-domain,LWD_ET,0.7702,0.7001,-0.0701,nnUNet
6,BraTS2024 in-domain,Dice_WT,0.8968,0.8455,-0.0512,nnUNet
7,BraTS2024 in-domain,Dice_TC,0.7017,0.5133,-0.1884,nnUNet
8,BraTS2024 in-domain,Dice_ET,0.7196,0.5235,-0.1960,nnUNet
9,BraTS2024 in-domain,LWD_WT,0.6855,0.5992,-0.0862,nnUNet


In [10]:
# Tabela 5: Visao pivot — uma linha por arquitetura+direcao, colunas por (cenario x metrica)

pivot = df_summary.copy()
pivot[["arquitetura", "subset"]] = pivot["cenario"].str.extract(r'(Baseline|MoME|nnUNet)_?(\d{4})?')
def short(name):
    if "in-domain" in name:
        if "Baseline_2020" in name: return "nnUNet@2020"
        if "Baseline_2024" in name: return "nnUNet@2024"
        if "MoME_2020"     in name: return "MoME@2020"
        if "MoME_2024"     in name: return "MoME@2024"
    if "nnUNet 2020 -> 2024" in name: return "nnUNet 2020->2024"
    if "nnUNet 2024 -> 2020" in name: return "nnUNet 2024->2020"
    if "MoME 2020 -> 2024"   in name: return "MoME 2020->2024"
    if "MoME 2024 -> 2020"   in name: return "MoME 2024->2020"
    return name

view = df_summary[["cenario", "Dice_WT", "Dice_TC", "Dice_ET", "LWD_WT", "LWD_TC", "LWD_ET"]].copy()
view["cenario"] = view["cenario"].apply(short)
view = view.set_index("cenario")
view.round(4)

,Dice_WT,Dice_TC,Dice_ET,LWD_WT,LWD_TC,LWD_ET
cenario,,,,,,
nnUNet@2020,0.9112,0.9013,0.8369,0.7151,0.8623,0.7702
nnUNet@2024,0.8968,0.7017,0.7196,0.6855,0.7128,0.7240
MoME@2020,0.8901,0.8637,0.7869,0.6153,0.7489,0.7001
MoME@2024,0.8455,0.5133,0.5235,0.5992,0.5199,0.5268
nnUNet 2020->2024,0.7414,0.4426,0.6045,0.5258,0.4111,0.6202
nnUNet 2024->2020,0.8811,0.7825,0.8127,0.6623,0.7462,0.7512
MoME 2020->2024,0.7411,0.3787,0.4922,0.5074,0.3144,0.4996
MoME 2024->2020,0.8320,0.7168,0.7706,0.5543,0.5518,0.5714
